In [1]:
%autosave 180
%load_ext autoreload
%autoreload 2

# import matplotlib
# #matplotlib.use('qtagg')
%matplotlib tk
# import matplotlib.pyplot as plt
# plt.ion()
# %matplotlib notebook

#|
import os
import numpy as np
import time
import nidaqmx
from nidaqmx.constants import (AcquisitionType)  # https://nidaqmx-python.readthedocs.io/en/latest/constants.html
import numpy as np
import matplotlib.pyplot as plt
from nidaqmx.constants import TerminalConfiguration
import math
from multiprocessing import Process

# 



Autosaving every 180 seconds


In [20]:
###################################################
###################################################
###################################################
fname_bmi = r'F:\bmi_tests\DON10798\22-07-24\calibration\Image_001_001.raw'
#fname_calibration = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

#
bmi_data = np.memmap(fname_bmi,
                   dtype='uint16',
                   mode='r',
                   shape=20000*512*512).reshape(20000,512,512)

# #
# calibration_data = np.memmap(fname_calibration,
#                    dtype='uint16',
#                    mode='r',
#                    shape=20000*512*512).reshape(20000,512,512)

# #
# print (bmi_data.shape, calibration_data.shape)

# #
# rois = np.load(r'D:\bmi\DON8498\22-06-08\rois_pixels_and_thresholds.npz')

# #
# contours = []
# for k in range(4):
#     contours.append(rois['cell'+str(k)+'_contour'])


In [22]:
#################################################
#################################################
#################################################
plt.figure()
ax=plt.subplot(111)
#plt.imshow(np.max(bmi_data[2000:5000],axis=0),
plt.imshow(bmi_data[2000:2500].mean(0),
          vmin=200,
          vmax=800)
# add contours on top of cells
#for c in range(len(contours)):
#    for k in range(len(contours[c])-1):
#        plt.plot([contours[c][k][0], contours[c][k+1][0]],
#                 [contours[c][k][1], contours[c][k + 1][1]],
#                           c='red')
# #                
# ax=plt.subplot(122)
# plt.imshow(calibration_data[2000:2500].mean(0))
# for c in range(len(contours)):
#     for k in range(len(contours[c])-1):
#         plt.plot([contours[c][k][0], contours[c][k+1][0]],
#                  [contours[c][k][1], contours[c][k + 1][1]],
#                            c='red')

plt.show()

In [6]:

#
shifts = np.load('/home/cat/shifts.npy')
corrs = np.load('/home/cat/corrs.npy')



#
plt.figure()
#plt.plot(shifts[:,0])
plt.plot(corrs)


plt.ylabel("Detected motion: x-axis (# pixels)")
plt.xlabel("Time (frames)")
plt.xlim(0,90000)
plt.show()

In [13]:
##########################################################
##########################################################
##########################################################
from tqdm import trange

#
def binarize(data):
    
    idx = np.where(data<2.5)[0]
    idx2 = np.where(data>=2.5)[0]
    
    data[idx] = 0
    data[idx2] = 1
    
    #
    return data

#
def get_velocity(rot1, rot2):
    
    # distance
    n_clicks_per_rotation = 500
    ball_diameter = 0.2  # distance in meters
    ball_circumference = ball_diameter*3.141592
    dist_per_click = ball_circumference/n_clicks_per_rotation
    
    # time
    sample_rate = 1000
    seconds_per_time_stamp = 1/sample_rate
    
    
    #
    bin1 = binarize(rot1)
    bin2 = binarize(rot2)
    
    #
    clicks = np.array((bin1, bin2)).T.squeeze()
    print (clicks)
        
    #
    vel = []
    times = []
    last_click_location = 0
    time_since_last_click = 0
    rot1_last_state = clicks[0,0]
    rot2_state = clicks[0,1]
    for k in trange(0,clicks.shape[0],1):
        
        #
        if clicks[k,0]!=rot1_last_state:
            distance = dist_per_click  #only walked 1 click
            time = time_since_last_click*seconds_per_time_stamp
            
            # 
            v = distance/time 
            
            #
            #print ("k: ", k, " last_click_location: ", last_click_location, " time, ", time_since_last_click)
            #print ("distance: ", distance, "  time: ", time)
            #print ("veL ", v)
            vel.append(v)
            times.append(k/sample_rate)       
            
            #
            time_since_last_click=0
            last_click_location=k #.copy()
            rot1_last_state=clicks[k,0]

        #    
        time_since_last_click+=1
        
    #
    return vel, times
    

#
data = np.load('/media/cat/4TB/donato/DON-009460/22-07-20/databmi_results.npz',
               allow_pickle=True)

#
lick_detector_times = data['lick_detector_abstime']
print ("lick detector: ", lick_detector_times.shape, lick_detector_times)

#
reward_times = data['reward_times'].squeeze().T[:,1]
print ("reward times: ", reward_times[:50])
ttl_times = data['ttl_times']
print ("ttl times: ", ttl_times)

#
rot1 = data['rotary_encoder1_abstime']
rot2 = data['rotary_encoder2_abstime']
print ("rotary 1: ", rot1)
print ("rotary 2: ", rot2)

#
vel, times = get_velocity(rot1,rot2)

lick detector:  (3009206, 1) [[5.19499007]
 [5.21086134]
 [5.22203601]
 ...
 [5.20195399]
 [5.20567888]
 [5.20924182]]
reward times:  [  435   930  1957  3142  4319  9248  9550 10624 11465 12081 13082 24338
 25180 26391 28298 34520 35206 35973 36970 56341 59727 61185 62203 66343
    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1    -1
    -1    -1]
ttl times:  [  28.8398133   28.8747537   28.9068659 ... 3028.7462991 3028.7802078
 3028.8124843]
rotary 1:  [[5.07255476]
 [5.08939769]
 [5.10008648]
 ...
 [0.07643473]
 [0.07643473]
 [0.07627279]]
rotary 2:  [[0.07740636]
 [0.0775683 ]
 [0.0775683 ]
 ...
 [5.078385  ]
 [5.08178598]
 [5.08567281]]
[[1. 0.]
 [1. 0.]
 [1. 0.]
 ...
 [0. 1.]
 [0. 1.]
 [0. 1.]]


100%|██████████| 3009206/3009206 [00:01<00:00, 1743755.16it/s]


In [14]:
plt.figure()
plt.plot(times, vel)
plt.show()

In [21]:
#######################################################
#######################################################
#######################################################
#
plt.figure()

# show lick detector deflections
sample_rate = 1000
idx = np.where(lick_detector_times<2)[0][::10]
licks = lick_detector_times[idx]
times = idx/sample_rate 
plt.scatter(times, 
            licks+1,
            label='lick detector (10x downsampled)')

# show lick detector deflections
idx = np.where(reward_times>0)[0]
temp = reward_times[idx]
plt.scatter(ttl_times[temp], 
            np.zeros(temp.shape[0])+2, 
            label='reward times')

#
plt.ylim(bottom=0)
plt.legend(fontsize=12)
plt.xlabel("Time (sec)")

#          
plt.show()

In [2]:
#####################################
#####################################
#####################################

#######################################
# FOR WINDOWS
# fname_root_path = r"D:\User Training"
# fname_fluorescence = r"D:\User Training\Readtest1\Image_001_001.raw"
# fname_freq =  r"F:\freq.npy"
# fname_rois = r"D:\User Training\rois.txt"

# FOR LINUX
fname_root_path = '/media/cat/4TB/donato/BSCOPE_tests/'
fname_fluorescence = os.path.join(fname_root_path, 
                                  'image_10000frames.raw')

#
fname_freq =  os.path.join(fname_root_path,
                           "freq.npy")

#
fname_rois = os.path.join(fname_root_path, 
                          "rois.txt")

# required for simulation mode
fname_ttl = os.path.join(fname_root_path,
                         "ttl_pulses.npy")


####################################################################### 			
################### DEFAULT PARAMTERS FOR BMI ######################### 			
####################################################################### 			
sampleRate_2P = 30
n_seconds_session = 10                          # number of seconds to run the BMI 
simulation_mode = True							# Run BMI in simulation mode (i.e. don't need Bscope input)

###############################################################
#################### INITIALIZE BMI ########################### 
###############################################################
bmi = BMI(simulation_mode,
          fname_root_path,
          fname_fluorescence,
          fname_rois,
          fname_freq,
          fname_ttl,
          sampleRate_2P,
          n_seconds_session)

# for simulation mode we sometimes want to slow down the processing;
# ... not as necessary 
bmi.sleep_time_sec = 0.033

# Flag to print out information from the proessing
bmi.verbose = False
bmi.verbose2 = False    # this displays the time it takes to copute ROI


 ROIS: ,  (10, 2)
   using square ROIs; TODO: use proper defined ROIs and cell masks ...
 ttl counter initialized:  [0] psm_f1d4b362
 tone frequency initialized:  [0] psm_244ef8f1


In [3]:
############################################################### 			
#################### INITIALIZE Plotting ###################### 
############################################################### 

#
plotter_ = Process(target=PlotROIs, args=(
                    bmi.shmem_rois_traces.name,
                    bmi.shmem_n_ttl.name,
                    bmi.rois_traces.shape,))

plotter_.start()

#
# plotter_.join()

# 
# plotter_.terminate()

...assuming sampling rate is 
 30 

In [4]:
############################################################### 			
#################### INITIALIZE TONE ########################## 
############################################################### 





In [5]:

#
bmi.run_BMI()


Running BMI (ctrl-c to stop)
TODO: implement shared memory to run plotting and tone playback directly from memroy
  more info:  https://docs.python.org/3/library/multiprocessing.shared_memory.html


% complete:  39%|##########4                | 116/300 [00:00<00:00, 1156.54it/s]

 duration to setup memmap:  0.00014853477478027344  sec.
     TODO: work with 1D flattened arrays


% complete:  89%|########################1  | 268/300 [00:00<00:00, 1366.92it/s]

...Saving BMI metadata...
... DONE BMI...


/home/cat/anaconda3/envs/bmi/lib/python3.8/site-packages/numpy/lib/npyio.py:713: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  val = np.asanyarray(val)
% complete: 100%|##########################9| 299/300 [00:19<00:00, 1366.92it/s]

In [18]:
# NOTES AND SCRATCH SPACE

In [13]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/100k_frames_ttl_data.zip')
print (data['data'].shape)

ttl_pulses = data['data']
np.save('/media/cat/4TB/donato/BSCOPE_tests/ttl_pulses.npy', ttl_pulses)

#
plt.figure()
plt.plot(data['data'])
plt.show()

(3599999, 1)


In [23]:
fname_fluorescence = '/media/cat/4TB/donato/BSCOPE_tests/8105/20220309/Image_001_001.raw'
n_frames_to_be_acquired = 1000
data_Ca = np.memmap(fname_fluorescence, dtype='uint16', mode='r', 
									   shape=(n_frames_to_be_acquired,512,512))
print (data_Ca.shape)
    
data_Ca.tofile('/media/cat/4TB/donato/BSCOPE_tests/image_1000frames.raw')

(1000, 512, 512)


In [27]:
###############################################################
###############################################################
###############################################################

#
def phase_correlation(a, b):
    G_a = np.fft.fft2(a)
    G_b = np.fft.fft2(b)
    conj_b = np.ma.conjugate(G_b)
    R = G_a * conj_b
    R /= np.absolute(R)
    r = np.fft.ifft2(R).real
    return r


def make_template_from_data(data,
                            n_imgs_to_sample=500,
                            n_best_imgs=100,
                            n_cores=1,
                            template=None):


    # find best correlation map first
    # don't pick random frames, much harder to find matchin frames
    random_img_sample_flag = False
    if random_img_sample_flag:
        idx_imgs = np.random.choice(np.arange(data.shape[0]),
                                    n_imgs_to_sample,
                                    replace=False)
    else:
        idx_start = np.random.choice(np.arange(data.shape[0] - n_imgs_to_sample))
        idx_imgs = np.arange(idx_start,
                             idx_start + n_imgs_to_sample,
                             1)

    #
    # print ("idx imgs; ", idx_imgs)

    # make temporary template to match to
    if template is None:
        template = np.mean(data[idx_imgs], axis=0)

    # parallelize
    if n_cores == 1:
        corr_maxs = np.zeros(idx_imgs.shape[0])
        ctr = 0
        for k in tqdm(idx_imgs, desc="computing phase correlations"):
            #
            temp = phase_correlation(data[k], template)

            r, c = np.unravel_index(temp.argmax(), temp.shape)

            maxcorr = temp[r, c]
            corr_maxs[ctr] = maxcorr
            ctr += 1
    else:
        # split the image indexes into gropus
        imgs_split = np.array_split(idx_imgs,
                                    40)

        #
        res = parmap.map(phase_correlation_parallel,
                         imgs_split,  # indexes of each image to process
                         template,  # defatul template
                         data,
                         pm_pbar=True,
                         pm_processes=n_cores
                         )

        # initialize arrays
        shifts = np.zeros((data.shape[0], 2))
        corr_maxs = np.zeros(data.shape[0])

        # merge the shifts:
        for k in range(len(res)):
            shifts = shifts + res[k][0]
            corr_maxs = corr_maxs + res[k][1]

        # select only the values chose
        # shifts = shifts[idx_imgs]
        corr_maxs = corr_maxs[idx_imgs]

    #
    idx = np.argsort(corr_maxs)[::-1]

    # take the n best images and recompute template
    idx_best = idx[:n_best_imgs]
    template = data[idx_imgs[idx_best]].mean(0)

    #
    return corr_maxs, template, idx_imgs



def phase_correlation_parallel2(idx_parmap,
                                template,
                                data):
    # load the data as mmap;
    # - this should avoid memmory crash issues
    # TODO: can even have this mmap reset every 1000 frames so that
    #   any size data can be processed!!
    #data = np.memmap(fname, dtype='uint16', mode='r')
    #data = data.reshape(-1, 512, 512)

    #
    corr_maxs = np.zeros(data.shape[0])
    shifts = np.zeros((data.shape[0], 2))

    #
    subtract_flag = True

    #
    a = template.copy()
    if idx_parmap[0] == 0:
        for idx in tqdm(idx_parmap, desc="phase corr computation"):

            # seelct an image
            # print ('idx: ', idx, data.shape)
            b = data[idx]

            #
            G_a = np.fft.fft2(a)
            G_b = np.fft.fft2(b)
            conj_b = np.ma.conjugate(G_b)
            R = G_a * conj_b
            R /= np.absolute(R)
            surface = np.fft.ifft2(R).real

            # compute peak location for row and column
            r, c = np.unravel_index(surface.argmax(), surface.shape)

            #
            corr_maxs[idx] = surface[r, c]

            # convert to roll function which has negative and positive values
            if subtract_flag:
                if r > 512 / 2:
                    r = r - 512

                if c > 512 / 2:
                    c = c - 512

            #
            shifts[idx] = [r, c]

    else:
        for idx in idx_parmap:

            # seelct an image
            # print ('idx: ', idx, data.shape)
            b = data[idx]

            #
            G_a = np.fft.fft2(a)
            G_b = np.fft.fft2(b)
            conj_b = np.ma.conjugate(G_b)
            R = G_a * conj_b
            R /= np.absolute(R)
            surface = np.fft.ifft2(R).real

            # compute peak location for row and column
            r, c = np.unravel_index(surface.argmax(), surface.shape)

            #
            corr_maxs[idx] = surface[r, c]

            # convert to roll function which has negative and positive values
            if subtract_flag:
                if r > 512 / 2:
                    r = r - 512

                if c > 512 / 2:
                    c = c - 512

            #
            shifts[idx] = [r, c]

    #
    return np.int32(shifts), corr_maxs
##########################################################
##########################################################
##########################################################
import parmap
fname_fluorescence = '/media/cat/4TB1/donato/DON-10798/22-07-26/data/Image_001_001.raw'

#
n_frames_for_template = 2500

#
data = np.memmap(fname_fluorescence, dtype='uint16', mode='r').reshape(-1, 512,512)
print (data.shape)
    
#
template = np.load('/media/cat/4TB1/donato/DON-10798/22-07-26/rois_pixels_and_thresholds.npz',
                   allow_pickle=True)['calibration_template']

#
plt.figure()
ax=plt.subplot(2,5,1)
shw = plt.imshow(template, 
                 vmin=0,
                 #vmax=3000
                )

#
n_imgs_to_sample=500
n_best_imgs = 100
n_cores = 1
                             
#                     
ctr=0
chunk_len = 10000
from tqdm import trange, tqdm
for k in range(0, data.shape[0], chunk_len):
    ax=plt.subplot(2,5,ctr+2)
    
    #
    temp = data[k:k+n_frames_for_template]
    
    #
    corr_maxs, template, idx_imgs = make_template_from_data(temp,
                                                            n_imgs_to_sample,
                                                            n_best_imgs,
                                                            n_cores)
    
    #
    shw = ax.imshow(template,
                    vmin=0,
                    vmax=2000
                   )
    #bar = plt.colorbar(shw)
    
    ctr+=1



(90000, 512, 512)


computing phase correlations: 100%|██████████| 500/500 [00:15<00:00, 32.43it/s]


In [35]:

import tifffile as tiff

fname = '/media/cat/4TB1/donato/DON-10798/22-07-26/data/Image_001_001.raw'
data = np.fromfile(fname,dtype=np.uint16).reshape(-1,512,512)
print (data.shape)

#
tiff.imwrite(fname.replace('.raw','.tiff'),data)

    

(90000, 512, 512)


In [2]:
data = np.load('/home/cat/rois_traces.npy')
print (data.shape)

(10, 600)


In [5]:
for k in range(data.shape[0]):
    plt.plot(data[k]+k*1000)
    
plt.show()

In [13]:
from scipy.signal import chirp, spectrogram

duration = 0.1
amp = 0.3
low = 2000
high = 18000

# TODO: Is there an easier way to generate tones on raw speakers???
fs = 200000  # 200kHz    (Sample Rate)
x = np.arange(int(fs * duration))
t = np.linspace(0, .1, 20000)

#
y2 = chirp(t, f0=low, f1=high, t1=duration, method='linear')*amp

plt.figure()
plt.plot(y2)
plt.show()
        

In [17]:
from scipy import stats
sample_rate = 200000
length_in_seconds = duration
amplitude = 0.3
noise = stats.truncnorm(-1, 1, 
                        scale=min(2**16, 2**amplitude)).rvs(int(sample_rate * length_in_seconds))


plt.figure()
plt.plot(noise)
plt.show()
        
